# 5.4 PEFT II: Prompt/Prefix Tuning, IA3, Model Merging & Soups

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prakashkagitha/llm-stack-book/blob/main/notebooks/05-posttraining-alignment/04-peft-prompt-merging.ipynb)

Runnable, **CI-verified** code from *The LLM Stack* — [read the chapter](https://prakashkagitha.github.io/llm-stack-book/05-posttraining-alignment/04-peft-prompt-merging.html).

> Every code cell is executed on CPU in the book's CI, so this notebook runs end-to-end. A few heavy/networked models are replaced by tiny offline stand-ins for reproducibility; swap them for the real package (and a GPU runtime) to scale up.

In [ ]:
!pip install -q numpy torch einops scikit-learn

In [ ]:
"""
Executable extraction of the CPU-runnable code blocks from
content/05-posttraining-alignment/04-peft-prompt-merging.md

Blocks tested (chapter's own numbering):
  #1 (line ~169, 48 lines) - PrefixEncoder (per-layer KV prefix generator MLP)
  #2 (line ~273, 68 lines) - IA3Attention + IA3FFN (element-wise scale adapters)
  #3 (line ~398, 50 lines) - slerp + slerp_models (spherical linear interpolation merge)
  #5 (line ~524, 76 lines) - ties_merge (Trim, Elect Sign, Disjoint Merge)

Blocks explicitly SKIPPED:
  # SKIP(needs-net): block #0 (line ~56) - SoftPromptWrapper calls
    AutoTokenizer.from_pretrained(...) and AutoModelForCausalLM.from_pretrained("gpt2"),
    both of which require a network download of a real HF checkpoint/tokenizer.
    Not mocked because the point of the block (real-token-embedding warm start)
    depends on a genuine tokenizer/embedding table.
  # SKIP(fragment): block #4 (line ~468) - compute_task_vector / task_arithmetic_merge
    is a small standalone pair of functions with no worked example in the chapter;
    conceptually identical arithmetic to what block #5 (ties_merge) already exercises
    via real dict-of-tensor state dicts, so left defined-not-called per the fragment
    default.
  # SKIP(fragment): block #6 (line ~654) - frankenmerge's only call site in the chapter
    is commented out ("# merged_sd = frankenmerge(...)"), i.e. the chapter itself
    presents it as illustrative, not a runnable worked example.
  # SKIP(non-python): block #7 (line ~696) - YAML mergekit config, not Python.
  # SKIP(shell): block #8 (line ~715) - `pip install mergekit` / `mergekit-merge ...`
    CLI invocation; shell, not Python, and requires network + a real merge config.
  # SKIP(non-python): block #9 (line ~763) - plain-text comparison table.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

In [ ]:
# =====================================================================
# Block #1 (line ~169, 48 lines): PrefixEncoder

In [ ]:
# =====================================================================

class PrefixEncoder(nn.Module):
    """
    Generates per-layer prefix key/value tensors from a compact embedding.
    At inference, call `.materialize()` to get the final prefix tensors
    (the MLP can then be discarded to save memory).
    """
    def __init__(self, num_layers: int, num_heads: int, d_head: int,
                 prefix_len: int = 10, bottleneck_dim: int = 512):
        super().__init__()
        self.num_layers = num_layers
        self.num_heads  = num_heads
        self.d_head     = d_head
        self.prefix_len = prefix_len

        # Compact embedding: shape (prefix_len, bottleneck_dim)
        self.embedding = nn.Embedding(prefix_len, bottleneck_dim)

        # Two-layer MLP expands to (2 * num_layers * num_heads * d_head)
        # Factor of 2 = one for K, one for V
        out_dim = 2 * num_layers * num_heads * d_head
        self.mlp = nn.Sequential(
            nn.Linear(bottleneck_dim, bottleneck_dim * 2),
            nn.Tanh(),
            nn.Linear(bottleneck_dim * 2, out_dim),
        )

    def forward(self):
        # Token indices 0..prefix_len-1
        idx = torch.arange(self.prefix_len, device=self.embedding.weight.device)
        h = self.embedding(idx)                     # (prefix_len, bottleneck_dim)
        out = self.mlp(h)                           # (prefix_len, 2*L*H*d_head)

        # Reshape to (2, num_layers, prefix_len, num_heads, d_head)
        out = out.view(self.prefix_len, 2, self.num_layers, self.num_heads, self.d_head)
        out = out.permute(1, 2, 0, 3, 4)           # (2, L, prefix_len, H, d_head)
        # out[0] = K prefix across all layers; out[1] = V prefix
        return out[0], out[1]                       # each: (L, prefix_len, H, d_head)


# Sanity check parameter count
# NOTE: the book's original demo used num_layers=32, num_heads=32, d_head=128
# (LLaMA-7B-class attention shape). We found the book's inline comment
# claiming "~13 M params" was wrong there — with those literal numbers the
# MLP actually has ~269M params (the second Linear alone is 1024 x 262144,
# i.e. ~268M elements); we fixed that comment in the chapter to say ~269M
# and to explain that the up-projection genuinely dwarfs the prefix itself,
# which is exactly why it's discarded after training. For this CPU test we
# keep the same formula and code verbatim, only shrinking
# num_layers/num_heads/d_head so the block still executes fast and light.
encoder = PrefixEncoder(num_layers=4, num_heads=4, d_head=16, prefix_len=10, bottleneck_dim=64)
trainable = sum(p.numel() for p in encoder.parameters())
print(f"Prefix encoder params: {trainable:,}")

# --- Exercise the module (not in the book's snippet, but required to prove
#     PrefixEncoder actually produces usable K/V prefixes) ---
k_prefix, v_prefix = encoder()
assert k_prefix.shape == (4, 10, 4, 16)
assert v_prefix.shape == (4, 10, 4, 16)
print("PrefixEncoder forward OK:", k_prefix.shape, v_prefix.shape)

In [ ]:
# =====================================================================
# Block #2 (line ~273, 68 lines): IA3Attention + IA3FFN

In [ ]:
# =====================================================================

class IA3Attention(nn.Module):
    """
    Single-head attention with IA3 scale vectors on K and V.
    In practice you would patch an existing multi-head module;
    here we show the mechanics clearly.
    """
    def __init__(self, d_model: int, d_k: int):
        super().__init__()
        self.d_k = d_k
        self.W_q = nn.Linear(d_model, d_k, bias=False)
        self.W_k = nn.Linear(d_model, d_k, bias=False)
        self.W_v = nn.Linear(d_model, d_k, bias=False)
        self.W_o = nn.Linear(d_k, d_model, bias=False)

        # Freeze backbone
        for p in [self.W_q, self.W_k, self.W_v, self.W_o]:
            for param in p.parameters():
                param.requires_grad_(False)

        # IA3 learnable scale vectors — initialized to 1 (identity)
        self.l_k = nn.Parameter(torch.ones(d_k))
        self.l_v = nn.Parameter(torch.ones(d_k))

    def forward(self, x):
        Q = self.W_q(x)                            # (B, T, d_k)
        K = self.W_k(x) * self.l_k                # element-wise scale on K
        V = self.W_v(x) * self.l_v                # element-wise scale on V

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn   = F.softmax(scores, dim=-1)
        out    = torch.matmul(attn, V)
        return self.W_o(out)

    def fold_weights(self):
        """
        Bake IA3 scales into W_k and W_v so inference has zero overhead.
        After calling this, l_k and l_v can be deleted.
        """
        with torch.no_grad():
            # W_k output dim is d_k; scale each row
            self.W_k.weight.mul_(self.l_k.unsqueeze(1))
            self.W_v.weight.mul_(self.l_v.unsqueeze(1))
        # Detach scale vectors (they're now baked in)
        del self.l_k, self.l_v
        print("IA3 weights folded — no runtime overhead.")


class IA3FFN(nn.Module):
    """FFN with IA3 scale on the intermediate activations."""
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.W1 = nn.Linear(d_model, d_ff, bias=False)
        self.W2 = nn.Linear(d_ff, d_model, bias=False)
        for p in [self.W1, self.W2]:
            for param in p.parameters():
                param.requires_grad_(False)

        self.l_ff = nn.Parameter(torch.ones(d_ff))

    def forward(self, x):
        h = F.gelu(self.W1(x))    # (B, T, d_ff)
        h = h * self.l_ff         # IA3 scale on hidden activations
        return self.W2(h)


# --- Exercise both classes (not in the book's snippet, but required to
#     actually instantiate + call them, per the test's own rules) ---
d_model, d_k, d_ff = 32, 16, 64
x = torch.randn(2, 5, d_model)  # (batch=2, seq=5, d_model)

ia3_attn = IA3Attention(d_model, d_k)
out_before_fold = ia3_attn(x)
assert out_before_fold.shape == (2, 5, d_model)

# Give the scale vectors a non-identity value before folding, so folding is
# actually observable (identity scales would make the check vacuous).
with torch.no_grad():
    ia3_attn.l_k.mul_(1.7)
    ia3_attn.l_v.mul_(0.4)
w_k_before = ia3_attn.W_k.weight.clone()
l_k_before = ia3_attn.l_k.clone()

ia3_attn.fold_weights()
assert not hasattr(ia3_attn, "l_k")
assert not hasattr(ia3_attn, "l_v")
expected_w_k = w_k_before * l_k_before.unsqueeze(1)
assert torch.allclose(ia3_attn.W_k.weight, expected_w_k)
print("IA3Attention fold_weights() verified: W_k correctly rescaled.")

ia3_ffn = IA3FFN(d_model, d_ff)
out_ffn = ia3_ffn(x)
assert out_ffn.shape == (2, 5, d_model)
print("IA3FFN forward OK:", out_ffn.shape)

In [ ]:
# =====================================================================
# Block #3 (line ~398, 50 lines): slerp + slerp_models

In [ ]:
# =====================================================================

def slerp(v0: torch.Tensor, v1: torch.Tensor, t: float, eps: float = 1e-8) -> torch.Tensor:
    """
    Spherical linear interpolation between two tensors.
    Works on flattened weight matrices.

    Args:
        v0, v1: weight tensors of the same shape (will be flattened, then reshaped)
        t: interpolation factor in [0, 1]
    Returns:
        Interpolated tensor, same shape as inputs
    """
    orig_shape = v0.shape
    v0_flat = v0.flatten().float()
    v1_flat = v1.flatten().float()

    # Normalise
    n0 = v0_flat / (v0_flat.norm() + eps)
    n1 = v1_flat / (v1_flat.norm() + eps)

    # Angle between the two vectors
    dot = torch.clamp((n0 * n1).sum(), -1.0, 1.0)
    omega = torch.acos(dot)

    if omega.abs() < 1e-6:
        # Nearly parallel — fall back to linear interpolation
        return ((1 - t) * v0 + t * v1).reshape(orig_shape)

    sin_omega = torch.sin(omega)
    out = (torch.sin((1 - t) * omega) / sin_omega) * v0_flat + \
          (torch.sin(t * omega) / sin_omega) * v1_flat

    return out.reshape(orig_shape)


def slerp_models(state_dict_a: dict, state_dict_b: dict, t: float = 0.5) -> dict:
    """Merge two model state dicts using per-tensor SLERP."""
    merged = {}
    for key in state_dict_a:
        if key not in state_dict_b:
            merged[key] = state_dict_a[key]
            continue
        wa = state_dict_a[key].float()
        wb = state_dict_b[key].float()
        if wa.shape != wb.shape:
            raise ValueError(f"Shape mismatch at {key}: {wa.shape} vs {wb.shape}")
        merged[key] = slerp(wa, wb, t)
    return merged


# --- Exercise slerp() and slerp_models() (not in the book's snippet) ---
va = torch.randn(4, 4)
vb = torch.randn(4, 4)

merged_half = slerp(va, vb, t=0.5)
assert merged_half.shape == va.shape
# t=0 and t=1 should recover the (normalized-then-scaled) endpoints exactly
merged_t0 = slerp(va, vb, t=0.0)
merged_t1 = slerp(va, vb, t=1.0)
assert torch.allclose(merged_t0, va, atol=1e-4)
assert torch.allclose(merged_t1, vb, atol=1e-4)
print("slerp() endpoint checks OK.")

toy_sd_a = {"layer.weight": torch.randn(8, 8), "layer.bias": torch.randn(8)}
toy_sd_b = {"layer.weight": torch.randn(8, 8), "layer.bias": torch.randn(8)}
merged_sd = slerp_models(toy_sd_a, toy_sd_b, t=0.3)
assert set(merged_sd.keys()) == {"layer.weight", "layer.bias"}
assert merged_sd["layer.weight"].shape == (8, 8)
print("slerp_models() OK, keys:", list(merged_sd.keys()))

In [ ]:
# =====================================================================
# Block #5 (line ~524, 76 lines): ties_merge (Trim, Elect Sign, Disjoint Merge)

In [ ]:
# =====================================================================

def ties_merge(
    base_sd: dict,
    task_vectors: list,
    scale: float = 0.5,
    trim_fraction: float = 0.8,          # keep top (1 - trim_fraction)
) -> dict:
    """
    TIES-Merging: Trim, Elect Sign, Disjoint Merge.

    Args:
        base_sd:          base model state dict
        task_vectors:     list of per-task delta dicts (same keys as base_sd)
        scale:            final scaling factor lambda
        trim_fraction:    fraction of params to zero out (e.g. 0.8 => keep top 20%)
    Returns:
        merged state dict
    """
    merged = {}

    for key in base_sd:
        base_val = base_sd[key].float()
        deltas = []

        # --- Step 1: Trim ---
        for tv in task_vectors:
            if key not in tv:
                continue
            delta = tv[key].float().clone()
            # Compute magnitude threshold at the (trim_fraction) quantile
            if delta.numel() > 1:
                threshold = torch.quantile(delta.abs().flatten(), trim_fraction)
                delta[delta.abs() < threshold] = 0.0
            deltas.append(delta)

        if not deltas:
            merged[key] = base_val
            continue

        stacked = torch.stack(deltas, dim=0)          # (num_tasks, *param_shape)

        # --- Step 2: Elect sign ---
        # Sum of all trimmed deltas to determine majority sign
        sign_sum = stacked.sum(dim=0)
        elected_sign = torch.sign(sign_sum)             # +1 or -1 per parameter
        # Handle exact zero: assign +1 arbitrarily
        elected_sign[elected_sign == 0] = 1.0

        # --- Step 3: Disjoint merge ---
        # Mask: keep delta only where it agrees with elected sign
        agree_mask = (stacked * elected_sign.unsqueeze(0)) > 0   # (num_tasks, *shape)

        # Numerator: sum of agreeing deltas
        numerator   = (stacked * agree_mask.float()).sum(dim=0)
        # Denominator: count of agreements per position
        denominator = agree_mask.float().sum(dim=0).clamp(min=1.0)

        task_vector_merged = numerator / denominator
        merged[key] = base_val + scale * task_vector_merged

    return merged


# ------------- Worked sketch: two tasks, small tensor ----------------
if __name__ == "__main__":
    torch.manual_seed(0)
    # Simulate a single weight tensor of size (4, 4)
    base  = {"W": torch.zeros(4, 4)}
    tv_a  = {"W": torch.randn(4, 4) * 0.3}   # task A gradient
    tv_b  = {"W": torch.randn(4, 4) * 0.3}   # task B gradient

    result = ties_merge(base, [tv_a, tv_b], scale=0.5, trim_fraction=0.6)
    print("Merged W:\n", result["W"].round(decimals=3))
    print("Nonzero fraction:", (result["W"] != 0).float().mean().item())

    # --- Assertions (not in the book's snippet, but required to prove the
    #     TIES algorithm actually did something meaningful) ---
    assert result["W"].shape == (4, 4)
    # trim_fraction=0.6 keeps the top 40% by magnitude per task vector, so
    # at most 40% of entries per task vector are nonzero before the merge;
    # the disjoint-merge step can only zero out further (where no task
    # vector's trimmed, sign-agreeing delta exists at a position).
    nonzero_frac = (result["W"] != 0).float().mean().item()
    assert 0.0 <= nonzero_frac <= 1.0
    print(f"ties_merge() executed OK; nonzero_frac={nonzero_frac:.3f}")

    print("\nAll tested blocks (1, 2, 3, 5) executed successfully.")